# arc3-agi Experiment Analysis

Explore and compare population evolution experiments stored in `experiments/runs.duckdb`.

**Quick start:**
1. Run cells in order from the top.
2. Change `EXPERIMENT_ID` / `EXPERIMENT_IDS` in the relevant cells to select what to plot.
3. All charts are interactive — zoom, pan, hover for exact values.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from arc3_agi.experiment import ExperimentStore, DEFAULT_DB_PATH

DB_PATH = DEFAULT_DB_PATH  # change to a custom path if needed
store = ExperimentStore(DB_PATH)
print(f"Database: {Path(DB_PATH).resolve()}")

## All experiments

In [ ]:
experiments = store.list_experiments()
display(experiments[['id', 'name', 'description', 'run_id', 'created_at', 'pop_count', 'gen_count']])

## Single-experiment analysis

Set `EXPERIMENT_ID` to the `id` from the table above.

In [ ]:
EXPERIMENT_ID = 1  # <-- change me

df = store.load_stats(EXPERIMENT_ID)
exp_name = experiments.loc[experiments['id'] == EXPERIMENT_ID, 'name'].iat[0]

print(f"Experiment '{exp_name}'  |  {df['pop_id'].nunique()} populations  |  {df['generation'].max()} generations")
df.head()

### Fan plot — mean fitness per population per generation

Each faint line is one population; the bold line is the mean across all populations.

In [ ]:
fig = go.Figure()

for pop_id, grp in df.groupby('pop_id'):
    fig.add_trace(go.Scatter(
        x=grp['generation'],
        y=grp['mean_fitness'],
        mode='lines',
        line=dict(color='steelblue', width=0.4),
        opacity=0.35,
        showlegend=False,
        hovertemplate=f'pop {pop_id}<br>gen %{{x}}<br>mean %{{y:.3f}}<extra></extra>',
    ))

mean_of_means = df.groupby('generation')['mean_fitness'].mean().reset_index()
fig.add_trace(go.Scatter(
    x=mean_of_means['generation'],
    y=mean_of_means['mean_fitness'],
    mode='lines',
    line=dict(color='darkblue', width=2.5),
    name='mean of means',
))

fig.update_layout(
    title=f"[{exp_name}] Mean fitness — all populations",
    xaxis_title='Generation',
    yaxis_title='Mean fitness',
    template='plotly_white',
    height=500,
)
fig.show()

### Max fitness fan plot

In [ ]:
fig = go.Figure()

for pop_id, grp in df.groupby('pop_id'):
    fig.add_trace(go.Scatter(
        x=grp['generation'],
        y=grp['max_fitness'],
        mode='lines',
        line=dict(color='tomato', width=0.4),
        opacity=0.35,
        showlegend=False,
        hovertemplate=f'pop {pop_id}<br>gen %{{x}}<br>max %{{y:.3f}}<extra></extra>',
    ))

mean_of_max = df.groupby('generation')['max_fitness'].mean().reset_index()
fig.add_trace(go.Scatter(
    x=mean_of_max['generation'],
    y=mean_of_max['max_fitness'],
    mode='lines',
    line=dict(color='darkred', width=2.5),
    name='mean of max',
))

fig.update_layout(
    title=f"[{exp_name}] Max fitness — all populations",
    xaxis_title='Generation',
    yaxis_title='Max fitness',
    template='plotly_white',
    height=500,
)
fig.show()

### Aggregate ribbon — mean ± 1 std across all populations

In [ ]:
agg = df.groupby('generation').agg(
    mean_mean=('mean_fitness', 'mean'),
    std_mean=('mean_fitness', 'std'),
    mean_max=('max_fitness', 'mean'),
    std_max=('max_fitness', 'std'),
).reset_index()

fig = go.Figure()

for label, col, colour in [
    ('Mean fitness', 'mean_mean', 'steelblue'),
    ('Max fitness',  'mean_max',  'tomato'),
]:
    std_col = col.replace('mean_', 'std_')
    upper = agg[col] + agg[std_col]
    lower = agg[col] - agg[std_col]
    fig.add_trace(go.Scatter(
        x=pd.concat([agg['generation'], agg['generation'][::-1]]),
        y=pd.concat([upper, lower[::-1]]),
        fill='toself',
        fillcolor=colour,
        opacity=0.15,
        line=dict(color='rgba(0,0,0,0)'),
        showlegend=False,
    ))
    fig.add_trace(go.Scatter(
        x=agg['generation'],
        y=agg[col],
        mode='lines',
        line=dict(color=colour, width=2),
        name=label,
    ))

fig.update_layout(
    title=f"[{exp_name}] Fitness ribbon (mean ± 1 std across populations)",
    xaxis_title='Generation',
    yaxis_title='Fitness',
    template='plotly_white',
    height=500,
)
fig.show()

### Generation wall-clock time

In [ ]:
time_agg = df.groupby('generation').agg(
    mean_dur=('duration_s', 'mean'),
    max_dur=('duration_s', 'max'),
).reset_index()

fig = px.line(
    time_agg,
    x='generation',
    y=['mean_dur', 'max_dur'],
    labels={'value': 'Duration (s)', 'generation': 'Generation', 'variable': 'Metric'},
    title=f"[{exp_name}] Generation wall-clock time",
    template='plotly_white',
)
fig.show()

---
## Multi-experiment comparison

Set `EXPERIMENT_IDS` to a list of ids to compare on the same chart.

In [ ]:
EXPERIMENT_IDS = [1, 2]  # <-- change me

exp_dfs = {}
exp_names = {}
for eid in EXPERIMENT_IDS:
    exp_dfs[eid] = store.load_stats(eid)
    row = experiments.loc[experiments['id'] == eid]
    exp_names[eid] = row['name'].iat[0] if not row.empty else str(eid)

print("Loaded:", {eid: exp_names[eid] for eid in EXPERIMENT_IDS})

### Overlay — mean-of-means fitness per experiment

In [ ]:
fig = go.Figure()

colours = px.colors.qualitative.Plotly

for i, eid in enumerate(EXPERIMENT_IDS):
    df_e = exp_dfs[eid]
    agg_e = df_e.groupby('generation').agg(
        mean_mean=('mean_fitness', 'mean'),
        std_mean=('mean_fitness', 'std'),
    ).reset_index()
    colour = colours[i % len(colours)]

    upper = agg_e['mean_mean'] + agg_e['std_mean']
    lower = agg_e['mean_mean'] - agg_e['std_mean']
    fig.add_trace(go.Scatter(
        x=pd.concat([agg_e['generation'], agg_e['generation'][::-1]]),
        y=pd.concat([upper, lower[::-1]]),
        fill='toself',
        fillcolor=colour,
        opacity=0.12,
        line=dict(color='rgba(0,0,0,0)'),
        showlegend=False,
    ))
    fig.add_trace(go.Scatter(
        x=agg_e['generation'],
        y=agg_e['mean_mean'],
        mode='lines',
        line=dict(color=colour, width=2),
        name=exp_names[eid],
    ))

fig.update_layout(
    title="Mean-of-means fitness comparison (ribbon = ±1 std across populations)",
    xaxis_title='Generation',
    yaxis_title='Mean fitness',
    template='plotly_white',
    height=500,
)
fig.show()

### Side-by-side fan plots

In [ ]:
n = len(EXPERIMENT_IDS)
fig = make_subplots(
    rows=1, cols=n,
    subplot_titles=[exp_names[eid] for eid in EXPERIMENT_IDS],
    shared_yaxes=True,
)

colours = px.colors.qualitative.Plotly

for col_idx, eid in enumerate(EXPERIMENT_IDS, start=1):
    df_e = exp_dfs[eid]
    colour = colours[(col_idx - 1) % len(colours)]

    for pop_id, grp in df_e.groupby('pop_id'):
        fig.add_trace(
            go.Scatter(
                x=grp['generation'],
                y=grp['mean_fitness'],
                mode='lines',
                line=dict(color=colour, width=0.4),
                opacity=0.3,
                showlegend=False,
                hovertemplate=f'pop {pop_id}<br>gen %{{x}}<br>mean %{{y:.3f}}<extra></extra>',
            ),
            row=1, col=col_idx,
        )

    mean_of_means = df_e.groupby('generation')['mean_fitness'].mean().reset_index()
    fig.add_trace(
        go.Scatter(
            x=mean_of_means['generation'],
            y=mean_of_means['mean_fitness'],
            mode='lines',
            line=dict(color=colour, width=2.5),
            name=exp_names[eid],
        ),
        row=1, col=col_idx,
    )

fig.update_layout(
    title='Side-by-side fan plots — mean fitness per population',
    template='plotly_white',
    height=500,
)
fig.update_xaxes(title_text='Generation')
fig.update_yaxes(title_text='Mean fitness', col=1)
fig.show()

### Overlay — max fitness comparison

In [ ]:
fig = go.Figure()

for i, eid in enumerate(EXPERIMENT_IDS):
    df_e = exp_dfs[eid]
    agg_e = df_e.groupby('generation').agg(
        mean_max=('max_fitness', 'mean'),
        std_max=('max_fitness', 'std'),
    ).reset_index()
    colour = colours[i % len(colours)]

    upper = agg_e['mean_max'] + agg_e['std_max']
    lower = agg_e['mean_max'] - agg_e['std_max']
    fig.add_trace(go.Scatter(
        x=pd.concat([agg_e['generation'], agg_e['generation'][::-1]]),
        y=pd.concat([upper, lower[::-1]]),
        fill='toself',
        fillcolor=colour,
        opacity=0.12,
        line=dict(color='rgba(0,0,0,0)'),
        showlegend=False,
    ))
    fig.add_trace(go.Scatter(
        x=agg_e['generation'],
        y=agg_e['mean_max'],
        mode='lines',
        line=dict(color=colour, width=2),
        name=exp_names[eid],
    ))

fig.update_layout(
    title="Max fitness comparison (ribbon = ±1 std across populations)",
    xaxis_title='Generation',
    yaxis_title='Max fitness',
    template='plotly_white',
    height=500,
)
fig.show()